# ⚡ TalkForge — AI Talking-Head Video Generator

Upload a portrait + audio → get a lip-synced talking video.

### Instructions
1. **Runtime → Change runtime type → T4 GPU** ← do this first!
2. **Runtime → Run all**
3. Wait ~8 minutes on first run
4. Click the `gradio.live` public URL printed at the bottom

> The notebook auto-restarts the kernel once after Cell 4. That is expected — Colab continues automatically.

In [ ]:
# CELL 1 — GPU check
import subprocess, sys
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print('\n'.join(r.stdout.split('\n')[:12]))
    print('\n✅ GPU detected — ready to proceed.')
else:
    print('❌  No GPU found.')
    print('   → Runtime → Change runtime type → T4 GPU → Save')
    print('   → Then Runtime → Disconnect and delete runtime → Run all')
    sys.exit(1)


In [ ]:
# CELL 2 — System packages
import subprocess
print('Installing system packages…')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run([
    'apt-get', 'install', '-y', '-qq',
    'ffmpeg', 'cmake', 'build-essential',
    'libboost-all-dev', 'libopenblas-dev',
    'liblapack-dev', 'libatlas-base-dev',
    'wget', 'curl', 'git', 'unzip'
], check=True)
r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('FFmpeg:', r.stdout.split('\n')[0])
print('✅ System packages ready.')


In [ ]:
# CELL 3 — Clone repos
import os, subprocess
TF_DIR = '/content/TalkForge'
ST_DIR = '/content/TalkForge/SadTalker'
if not os.path.exists(os.path.join(TF_DIR, 'main.py')):
    print('Cloning TalkForge…')
    subprocess.run(['git', 'clone', '--depth', '1',
        'https://github.com/rydenxgod-bot/TalkForge.git', TF_DIR], check=True)
else:
    print('✓ TalkForge already present')
if not os.path.exists(os.path.join(ST_DIR, 'inference.py')):
    print('Cloning SadTalker…')
    subprocess.run(['git', 'clone', '--depth', '1',
        'https://github.com/OpenTalker/SadTalker.git', ST_DIR], check=True)
else:
    print('✓ SadTalker already present')
print('✅ Repos ready.')


In [ ]:
# CELL 4 — Install Python packages + kernel restart
#
# We do NOT install SadTalker requirements.txt because it pins numpy<2
# which causes ABI crashes with Colab's pre-compiled pandas/gradio.
# We keep numpy 2.x and patch deprecated calls in Cell 5 instead.
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + list(args), check=True)

print('[1/7] huggingface_hub…')
pip('huggingface_hub')
print('[2/7] torch (cu121)…')
pip('torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121')
print('[3/7] gradio + numpy 2.x…')
pip('gradio==4.44.1', 'numpy>=2.0')
print('[4/7] audio / vision…')
pip('librosa', 'soundfile', 'pillow', 'opencv-python-headless', 'imageio', 'imageio-ffmpeg', 'scipy', 'tqdm', 'pyyaml')
print('[5/7] face analysis…')
pip('face_alignment', 'facexlib', 'gfpgan', 'realesrgan', 'basicsr')
print('[6/7] model utilities…')
pip('safetensors', 'kornia', 'einops', 'yacs', 'batch-face', 'mediapipe')
print('[7/7] dlib (compiles ~3 min)…')
pip('dlib')
print()
print('✅ All packages installed.')
print()
print('━' * 52)
print('  🔄  Restarting kernel to flush ABI-incompatible modules…')
print('  Colab will automatically continue with Cell 5.')
print('━' * 52)
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)


In [ ]:
# CELL 5 — Patch SadTalker + basicsr for numpy 2.x
import os, re
ALIASES = [
    (r'\bnp\.bool\b',    'np.bool_'),
    (r'\bnp\.int\b',     'np.int_'),
    (r'\bnp\.float\b',   'np.float64'),
    (r'\bnp\.complex\b', 'np.complex128'),
    (r'\bnp\.object\b',  'np.object_'),
    (r'\bnp\.str\b',     'np.str_'),
]
def patch_dir(root_dir, label):
    count = 0
    for root, _, files in os.walk(root_dir):
        for fname in files:
            if not fname.endswith('.py'):
                continue
            fpath = os.path.join(root, fname)
            try:
                src = open(fpath, 'r', encoding='utf-8', errors='ignore').read()
                new = src
                for old, rep in ALIASES:
                    new = re.sub(old, rep, new)
                if new != src:
                    open(fpath, 'w', encoding='utf-8').write(new)
                    count += 1
            except Exception:
                pass
    print(f'  {label}: patched {count} file(s)')
print('Applying numpy 2.x patches…')
patch_dir('/content/TalkForge/SadTalker', 'SadTalker')
try:
    import pkg_resources
    bsr = pkg_resources.get_distribution('basicsr').location
    patch_dir(os.path.join(bsr, 'basicsr'), 'basicsr')
except Exception as e:
    print(f'  basicsr skipped: {e}')
try:
    fa = pkg_resources.get_distribution('face_alignment').location
    patch_dir(os.path.join(fa, 'face_alignment'), 'face_alignment')
except Exception as e:
    print(f'  face_alignment skipped: {e}')
print('✅ Patches applied.')


In [ ]:
# CELL 6 — Download SadTalker weights via huggingface_hub
#
# Uses hf_hub_download which handles LFS pointers automatically.
# Sources:
#   safetensors + mapping  → vinthony/SadTalker-V002rc
#   pth core files         → camenduru/SadTalker
#   BFM + hub + gfpgan     → barisaydin/sadtalker
import os, shutil, zipfile
from huggingface_hub import hf_hub_download

ST_DIR   = '/content/TalkForge/SadTalker'
CKPT_DIR = os.path.join(ST_DIR, 'checkpoints')
W_DIR    = '/content/TalkForge/weights'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(W_DIR,    exist_ok=True)

def dl(repo_id, filename, dest_name=None):
    dest_name = dest_name or os.path.basename(filename)
    dest = os.path.join(CKPT_DIR, dest_name)
    if os.path.exists(dest) and os.path.getsize(dest) > 100_000:
        print(f'  ✓ {dest_name}  ({os.path.getsize(dest)//1_000_000} MB)')
        return
    print(f'  ↓ {dest_name}  [{repo_id}]')
    cached = hf_hub_download(repo_id=repo_id, filename=filename,
                              repo_type='model', cache_dir='/tmp/hf_cache')
    shutil.copy2(cached, dest)
    shutil.copy2(cached, os.path.join(W_DIR, dest_name))
    print(f'    → {os.path.getsize(dest)//1_000_000} MB')

print('--- Safetensor checkpoints ---')
dl('vinthony/SadTalker-V002rc', 'SadTalker_V0.0.2_256.safetensors')
dl('vinthony/SadTalker-V002rc', 'SadTalker_V0.0.2_512.safetensors')
dl('vinthony/SadTalker-V002rc', 'mapping_00109-model.pth.tar')
dl('vinthony/SadTalker-V002rc', 'mapping_00229-model.pth.tar')

print('--- Core model weights ---')
dl('camenduru/SadTalker', 'auido2exp_00300-model.pth')
dl('camenduru/SadTalker', 'auido2pose_00140-model.pth')
dl('camenduru/SadTalker', 'epoch_20.pth')
dl('camenduru/SadTalker', 'facevid2vid_00189-model.pth.tar')
dl('camenduru/SadTalker', 'wav2lip.pth')
dl('camenduru/SadTalker', 'shape_predictor_68_face_landmarks.dat')

print('--- BFM Fitting data ---')
BFM_DIR = os.path.join(CKPT_DIR, 'BFM_Fitting')
if not os.path.exists(BFM_DIR):
    bfm_zip = hf_hub_download(repo_id='barisaydin/sadtalker',
                               filename='checkpoints/BFM_Fitting.zip',
                               repo_type='model', cache_dir='/tmp/hf_cache')
    with zipfile.ZipFile(bfm_zip, 'r') as z:
        members = [m for m in z.namelist() if 'BFM_Fitting' in m]
        z.extractall(CKPT_DIR, members=members)
    # normalise path in case zip extracted nested
    for root, dirs, _ in os.walk(CKPT_DIR):
        if 'BFM_Fitting' in dirs:
            found = os.path.join(root, 'BFM_Fitting')
            if found != BFM_DIR:
                shutil.move(found, BFM_DIR)
            break
    print(f'  ✓ BFM_Fitting extracted')
else:
    print('  ✓ BFM_Fitting')

print('--- Face alignment hub models ---')
HUB_DIR = os.path.join(CKPT_DIR, 'hub', 'checkpoints')
os.makedirs(HUB_DIR, exist_ok=True)
for hfname, dname in [
    ('checkpoints/hub/checkpoints/2DFAN4-cd938726ad.zip', '2DFAN4-cd938726ad.zip'),
    ('checkpoints/hub/checkpoints/s3fd-619a316812.pth',   's3fd-619a316812.pth'),
]:
    dest = os.path.join(HUB_DIR, dname)
    if not os.path.exists(dest):
        cached = hf_hub_download(repo_id='barisaydin/sadtalker', filename=hfname,
                                  repo_type='model', cache_dir='/tmp/hf_cache')
        shutil.copy2(cached, dest)
        print(f'  ✓ {dname}')
    else:
        print(f'  ✓ {dname}')

import torch
torch.hub.set_dir(os.path.join(CKPT_DIR, 'hub'))
os.environ['TORCH_HOME'] = CKPT_DIR
print()
print('✅ All SadTalker weights ready.')


In [ ]:
# CELL 7 — Download GFPGAN face-enhancer weights
import os, shutil
from huggingface_hub import hf_hub_download

ST_DIR  = '/content/TalkForge/SadTalker'
GFP_DIR = os.path.join(ST_DIR, 'gfpgan', 'weights')
os.makedirs(GFP_DIR, exist_ok=True)

GFP_FILES = [
    ('barisaydin/sadtalker', 'gfpgan/weights/GFPGANv1.4.pth',              'GFPGANv1.4.pth'),
    ('barisaydin/sadtalker', 'gfpgan/weights/detection_Resnet50_Final.pth', 'detection_Resnet50_Final.pth'),
    ('barisaydin/sadtalker', 'gfpgan/weights/parsing_parsenet.pth',         'parsing_parsenet.pth'),
    ('barisaydin/sadtalker', 'gfpgan/weights/alignment_WFLW_4HG.pth',       'alignment_WFLW_4HG.pth'),
]

print('Downloading GFPGAN weights…')
for repo, fname, dname in GFP_FILES:
    dest = os.path.join(GFP_DIR, dname)
    if os.path.exists(dest) and os.path.getsize(dest) > 100_000:
        print(f'  ✓ {dname}')
        continue
    print(f'  ↓ {dname}')
    cached = hf_hub_download(repo_id=repo, filename=fname,
                              repo_type='model', cache_dir='/tmp/hf_cache')
    shutil.copy2(cached, dest)
    print(f'    → {os.path.getsize(dest)//1_000_000} MB')

FX_CACHE = os.path.expanduser('~/.cache/facexlib')
os.makedirs(FX_CACHE, exist_ok=True)
for fname in ('detection_Resnet50_Final.pth', 'parsing_parsenet.pth'):
    src = os.path.join(GFP_DIR, fname)
    dst = os.path.join(FX_CACHE, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
print()
print('✅ GFPGAN weights ready.')


In [ ]:
# CELL 8 — Final smoke test
import sys, os
TF_DIR = '/content/TalkForge'
ST_DIR = '/content/TalkForge/SadTalker'
for p in [TF_DIR, ST_DIR]:
    if p not in sys.path: sys.path.insert(0, p)
os.chdir(TF_DIR)
all_ok = True
IMPORT_TESTS = [
    ('numpy',         'import numpy as np; print(f"  numpy {np.__version__}")'),
    ('torch',         'import torch; print(f"  torch {torch.__version__}, CUDA={torch.cuda.is_available()}")'),
    ('gradio',        'import gradio; print(f"  gradio {gradio.__version__}")'),
    ('cv2',           'import cv2; print(f"  cv2 {cv2.__version__}")'),
    ('librosa',       'import librosa; print(f"  librosa {librosa.__version__}")'),
    ('face_alignment','import face_alignment; print("  face_alignment OK")'),
    ('gfpgan',        'import gfpgan; print("  gfpgan OK")'),
    ('basicsr',       'import basicsr; print("  basicsr OK")'),
    ('safetensors',   'import safetensors; print("  safetensors OK")'),
]
print('Import checks:')
for name, stmt in IMPORT_TESTS:
    try: exec(stmt)
    except Exception as e:
        print(f'  ✗ {name}: {e}')
        all_ok = False
print('\nWeight checks:')
CKPT = '/content/TalkForge/SadTalker/checkpoints'
REQUIRED = [
    f'{CKPT}/SadTalker_V0.0.2_256.safetensors',
    f'{CKPT}/auido2exp_00300-model.pth',
    f'{CKPT}/auido2pose_00140-model.pth',
    f'{CKPT}/epoch_20.pth',
    f'{CKPT}/facevid2vid_00189-model.pth.tar',
    f'{CKPT}/BFM_Fitting',
    '/content/TalkForge/SadTalker/gfpgan/weights/GFPGANv1.4.pth',
    '/content/TalkForge/SadTalker/gfpgan/weights/detection_Resnet50_Final.pth',
]
for f in REQUIRED:
    exists = os.path.exists(f)
    sz = ''
    if exists and os.path.isfile(f): sz = f' ({os.path.getsize(f)//1_000_000} MB)'
    print(f'  {chr(10003) if exists else chr(10007)} {os.path.basename(f)}{sz}')
    if not exists: all_ok = False
print()
if all_ok:
    print('✅ All checks passed — ready to launch!')
else:
    print('⚠️  Some checks failed.')
    print('   Import failures → Runtime → Restart session → re-run from Cell 5')
    print('   Missing weights  → re-run Cells 6 and 7')


In [ ]:
# CELL 9 — Configure runtime paths
import os, torch
CKPT_DIR = '/content/TalkForge/SadTalker/checkpoints'
torch.hub.set_dir(os.path.join(CKPT_DIR, 'hub'))
os.environ['TORCH_HOME']          = CKPT_DIR
os.environ['FACEXLIB_MODEL_PATH'] = '/content/TalkForge/SadTalker/gfpgan/weights'
f256 = os.path.join(CKPT_DIR, 'SadTalker_V0.0.2_256.safetensors')
sz = os.path.getsize(f256) if os.path.exists(f256) else 0
if sz > 100_000_000:
    print(f'✓ 256 safetensors: {sz//1_000_000} MB')
else:
    print(f'⚠️  256 safetensors looks wrong ({sz} bytes) — re-run Cell 6')
print('✅ Runtime paths configured.')


In [ ]:
# CELL 10 — 🚀 Launch TalkForge
import sys, os, warnings, torch
TF_DIR = '/content/TalkForge'
ST_DIR = '/content/TalkForge/SadTalker'
for p in [TF_DIR, ST_DIR]:
    if p not in sys.path: sys.path.insert(0, p)
os.chdir(TF_DIR)
os.makedirs(os.path.join(TF_DIR, 'outputs'), exist_ok=True)
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTHONWARNINGS']        = 'ignore'
CKPT_DIR = '/content/TalkForge/SadTalker/checkpoints'
torch.hub.set_dir(os.path.join(CKPT_DIR, 'hub'))
os.environ['TORCH_HOME']          = CKPT_DIR
os.environ['FACEXLIB_MODEL_PATH'] = '/content/TalkForge/SadTalker/gfpgan/weights'
print('═' * 60)
print('  ⚡  TalkForge — starting Gradio…')
print('  Public URL appears below in ~20 seconds.')
print('  The link works for 72 hours — share it with anyone.')
print('═' * 60)
from app.ui import launch
launch(share=True, port=7860)
